# Section 4.4 — Expert Withdrawal
## Task 1: binary walking detection

Part A is the cutoff experiment exactly as written: fused self-teaching during
training, camera withdrawn at $k_c = 1250$, microphones inverted and averaged
thereafter with the parameters frozen at $k_c$.

Part B adds no computation to the experiment. It reads the arrays the loop has
already produced and reports the quantities the chapter's outstanding-items
list asks for: the expert baseline on each segment, the class distribution and
transition count of each segment, per-class recall before and after the
cutoff, and the two figures.

## Part A — the experiment

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import seaborn as sns
from sklearn.metrics import classification_report

# ------------------ STYLE ------------------
sns.set(style="whitegrid")
sns.set_palette("deep")
sns.set(font="DejaVu Sans")
np.random.seed(1)

# ==========================
# SETTINGS
# ==========================
CSV_PATH = "../data/binary_walking_1851.csv"
K_CUT = 1250
eps = 1e-9

# ------------------ LOAD DATA ------------------
df = pd.read_csv(CSV_PATH)

# add second inexpert column here
needed = ["human", "mouse", "dBFS_mic1", "dBFS_mic2"]
df = df[needed].dropna().reset_index(drop=True)

# ------------------ BUILD MANUAL GT FROM MOUSE ------------------
m = df["mouse"].astype(int)
m = m.mask((m == 1) & (m.shift(1, fill_value=0) == 1), 0)
walking_gt = (m.cumsum() % 2).astype(int)
df["walking_gt"] = walking_gt

Np = len(df)
if K_CUT >= Np - 1:
    raise ValueError(f"K_CUT={K_CUT} must be < Np-1={Np-1}")

# ------------------ SIGNALS ------------------
gt = np.where(df["walking_gt"].to_numpy() == 1, 1, -1).astype(int)

# Expert measurement (camera) as ±1 (USED ONLY BEFORE CUT)
y1 = np.where(df["human"].astype(int).to_numpy() == 1, 1.0, -1.0)

# Inexpert measurement 1
y0_raw = df["dBFS_mic1"].astype(float).to_numpy()
y0 = y0_raw - np.mean(y0_raw)

# Inexpert measurement 2
y2_raw = df["dBFS_mic2"].astype(float).to_numpy()
y2 = y2_raw - np.mean(y2_raw)

# ------------------ KF PARAMETERS (UNCHANGED) ------------------
sig1 = 1.0
c1 = 1.0

# ------------------ STORAGE ------------------
Xki_all = np.zeros(Np, dtype=int)

c_hat_hist_0 = np.zeros(Np)
sig2_hat_hist_0 = np.zeros(Np)

c_hat_hist_2 = np.zeros(Np)
sig2_hat_hist_2 = np.zeros(Np)

# ------------------ INITIALIZATION (UNCHANGED) ------------------
X_updated_all = np.array([0.0])
P_all = np.eye(1)
Q_all = 10 * np.eye(1)
A_all = np.eye(1)

# inexpert parameters
c_hat_0 = 1.0
sig2_hat_0 = 1.0

c_hat_2 = 1.0
sig2_hat_2 = 1.0

# cutoff freeze
cut_done = False

c_frozen_0 = None
sig2_frozen_0 = None

c_frozen_2 = None
sig2_frozen_2 = None

baseline_frozen_0 = 0.0
baseline_frozen_2 = 0.0

In [2]:
# ------------------ MAIN LOOP ------------------
for k in range(Np):

    # ==========================
    # TRAIN: k <= K_CUT
    # ==========================
    if k <= K_CUT:

        sig2_hat_0 = max(float(sig2_hat_0), 1e-6)
        sig2_hat_2 = max(float(sig2_hat_2), 1e-6)

        R_all = np.array([
            [sig1**2,     0.0,         0.0],
            [0.0,     sig2_hat_0,      0.0],
            [0.0,         0.0,    sig2_hat_2]
        ])

        C_all = np.array([
            [c1],
            [c_hat_0],
            [c_hat_2]
        ])

        Xm = A_all @ X_updated_all.reshape(-1, 1)
        Pm = A_all @ P_all @ A_all.T + Q_all

        residual_vec = np.array([y1[k], y0[k], y2[k]]) - (C_all @ Xm).flatten()

        K = Pm @ C_all.T @ np.linalg.inv(C_all @ Pm @ C_all.T + R_all)
        X_updated_all = (Xm + K @ residual_vec.reshape(-1, 1)).flatten()
        P_all = (np.eye(1) - K @ C_all) @ Pm

        Xki_all[k] = -1 if X_updated_all[0] < 0 else 1

        #-------- learning inexpert 1 exactly like before --------
        denom = Xki_all[:k+1].astype(float)
        denom[np.abs(denom) < 1e-6] = 1.0

        c_hat_0 = float(np.mean(y0[:k+1] / denom))
        sig2_hat_0 = float(np.var(y0[:k+1] - c_hat_0 * Xki_all[:k+1]))
        sig2_hat_0 = max(sig2_hat_0, 1e-6)

        # -------- learning inexpert 2 the same way --------
        c_hat_2 = float(np.mean(y2[:k+1] / denom))
        sig2_hat_2 = float(np.var(y2[:k+1] - c_hat_2 * Xki_all[:k+1]))
        sig2_hat_2 = max(sig2_hat_2, 1e-6)

        c_hat_hist_0[k] = c_hat_0
        sig2_hat_hist_0[k] = sig2_hat_0

        c_hat_hist_2[k] = c_hat_2
        sig2_hat_hist_2[k] = sig2_hat_2

        #learning from ground truth
        # denom = gt[:k+1].astype(float)
        # denom[np.abs(denom) < 1e-6] = 1.0

        # c_hat_0 = float(np.mean(y0[:k+1] / denom))
        # sig2_hat_0 = float(np.var(y0[:k+1] - c_hat_0 * gt[:k+1]))
        # sig2_hat_0 = max(sig2_hat_0, 1e-6)

        # c_hat_2 = float(np.mean(y2[:k+1] / denom))
        # sig2_hat_2 = float(np.var(y2[:k+1] - c_hat_2 * gt[:k+1]))
        # sig2_hat_2 = max(sig2_hat_2, 1e-6)


        # c_hat_hist_0[k] = c_hat_0
        # sig2_hat_hist_0[k] = sig2_hat_0

        # c_hat_hist_2[k] = c_hat_2
        # sig2_hat_hist_2[k] = sig2_hat_2

        if (not cut_done) and (k == K_CUT):
            cut_done = True

            c_frozen_0 = float(c_hat_0)
            sig2_frozen_0 = float(sig2_hat_0)

            c_frozen_2 = float(c_hat_2)
            sig2_frozen_2 = float(sig2_hat_2)

            print(f"\n=== CUT at k={K_CUT} ===")
            print(f"Frozen c_hat_0       = {c_frozen_0:.6f}")
            print(f"Frozen sigma^2_hat_0 = {sig2_frozen_0:.6f}")
            print(f"Frozen c_hat_2       = {c_frozen_2:.6f}")
            print(f"Frozen sigma^2_hat_2 = {sig2_frozen_2:.6f}")
            print("DEPLOY starts next sample: camera OFF, two inexperts only.\n")

    # ==========================
    # DEPLOY: k > K_CUT
    # ==========================
    else:
        y0c_now = y0[k] - baseline_frozen_0
        y2c_now = y2[k] - baseline_frozen_2

        x_hat_0 = y0c_now / (float(c_frozen_0) + eps)
        x_hat_2 = y2c_now / (float(c_frozen_2) + eps)

        # combine both inexperts, no other logic changed
        x_hat = (x_hat_0 + x_hat_2) / 2.0

        Xki_all[k] = -1 if x_hat < 0 else 1

        c_hat_hist_0[k] = c_frozen_0
        sig2_hat_hist_0[k] = sig2_frozen_0

        c_hat_hist_2[k] = c_frozen_2
        sig2_hat_hist_2[k] = sig2_frozen_2


=== CUT at k=1250 ===
Frozen c_hat_0       = 4.096334
Frozen sigma^2_hat_0 = 10.998078
Frozen c_hat_2       = 6.131768
Frozen sigma^2_hat_2 = 6.130632
DEPLOY starts next sample: camera OFF, two inexperts only.



In [3]:
# ---------- sensitivity of the result to the cutoff index ----------
# Part A logic unchanged; only k_cut varies.

def run_cutoff(k_cut):
    Xk = np.zeros(Np, dtype=int)
    Xu = np.array([0.0]); P = np.eye(1)
    Q  = 10 * np.eye(1);  A = np.eye(1)
    ch0 = ch2 = 1.0
    r0  = r2  = 1.0
    frozen = None

    for k in range(Np):
        if k <= k_cut:
            r0 = max(float(r0), 1e-6); r2 = max(float(r2), 1e-6)
            R = np.array([[sig1**2, 0.0, 0.0], [0.0, r0, 0.0], [0.0, 0.0, r2]])
            C = np.array([[c1], [ch0], [ch2]])
            Xm = A @ Xu.reshape(-1, 1)
            Pm = A @ P @ A.T + Q
            rv = np.array([y1[k], y0[k], y2[k]]) - (C @ Xm).flatten()
            K  = Pm @ C.T @ np.linalg.inv(C @ Pm @ C.T + R)
            Xu = (Xm + K @ rv.reshape(-1, 1)).flatten()
            P  = (np.eye(1) - K @ C) @ Pm
            Xk[k] = -1 if Xu[0] < 0 else 1

            d = Xk[:k+1].astype(float); d[np.abs(d) < 1e-6] = 1.0
            ch0 = float(np.mean(y0[:k+1] / d))
            r0  = max(float(np.var(y0[:k+1] - ch0 * Xk[:k+1])), 1e-6)
            ch2 = float(np.mean(y2[:k+1] / d))
            r2  = max(float(np.var(y2[:k+1] - ch2 * Xk[:k+1])), 1e-6)
            if k == k_cut:
                frozen = (ch0, r0, ch2, r2)
        else:
            xh = 0.5 * (y0[k] / (frozen[0] + eps) + y2[k] / (frozen[2] + eps))
            Xk[k] = -1 if xh < 0 else 1

    e = (Xk != gt)
    return (100 * e[:k_cut+1].mean(), 100 * e[k_cut+1:].mean(),
            int(e[k_cut+1:].sum()), frozen)


# Define 'cam' here, as it's used in the loop below
cam = np.where(y1 > 0, 1, -1)

grid = [200, 300, 400, 600, 800, 1000, 1250, 1500, 1600]
rows = []
print(f"{'k_c':>6}{'train':>9}{'deploy':>9}{'errors':>8}{'n_dep':>7}"
      f"{'expert on dep':>15}{'c_1':>9}{'c_2':>9}")
for kc in grid:
    tr_e, dp_e, n_err, fz = run_cutoff(kc)
    exp_e = 100 * np.mean(cam[kc+1:] != gt[kc+1:])
    rows.append((kc, tr_e, dp_e, exp_e))
    print(f"{kc:>6}{tr_e:>8.2f}%{dp_e:>8.2f}%{n_err:>8}{Np-kc-1:>7}"
          f"{exp_e:>14.2f}%{fz[0]:>9.4f}{fz[2]:>9.4f}")

kcs   = [r[0] for r in rows]
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(kcs, [r[3] for r in rows], "--", color="gray", marker="o",
        markersize=4, label="Expert alone, deployment segment")
ax.plot(kcs, [r[2] for r in rows], color="steelblue", marker="o",
        markersize=4, label="Microphones alone, deployment segment")
ax.set_xlabel("Cutoff index $k_c$")
ax.set_ylabel("Deployment error [%]")
ax.set_yscale("log")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("ch5_binary_kc_sweep.pdf", bbox_inches="tight")
plt.show()

   k_c    train   deploy  errors  n_dep  expert on dep      c_1      c_2


   200    2.99%    1.27%      21   1650         29.03%   3.9853   6.0184


   300    2.66%    1.29%      20   1550         28.97%   4.1328   6.2429


   400    2.24%    1.31%      19   1450         30.07%   4.2077   6.0986


   600    2.00%    1.28%      16   1250         30.00%   4.1306   6.1953
   800    1.87%    1.14%      12   1050         28.19%   4.3278   6.3075


  1000    1.60%    1.18%      10    850         30.12%   4.2071   6.1104


  1250    1.52%    1.00%       6    600         31.17%   4.0963   6.1318


  1500    1.40%    0.86%       3    350         40.00%   4.0348   6.0296


  1600    1.44%    1.20%       3    250         30.00%   4.1177   6.0404


/tmp/ipykernel_927/2220138252.py:67: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# ------------------ ERRORS ------------------
err = (Xki_all != gt).astype(int)

err_before = 100 * np.mean(err[:K_CUT+1])
err_after  = 100 * np.mean(err[K_CUT+1:])

cum_err_overall = 100.0 * np.cumsum(err) / (np.arange(Np) + 1)

err_deploy = err[K_CUT+1:]
cum_err_deploy = 100.0 * np.cumsum(err_deploy) / (np.arange(len(err_deploy)) + 1)

print(f"Error BEFORE cut (TRAIN): {err_before:.2f}%")
print(f"Error AFTER  cut (DEPLOY two-inexpert): {err_after:.2f}%")

Error BEFORE cut (TRAIN): 1.52%
Error AFTER  cut (DEPLOY two-inexpert): 1.00%


In [5]:
# ==========================
# PLOTS
# ==========================

# 1) Overall cumulative error
plt.figure(figsize=(11, 4))
plt.plot(cum_err_overall, label=f"Overall cumulative (final {cum_err_overall[-1]:.2f}%)")
plt.axvline(K_CUT, linestyle="--", linewidth=2, label=f"CUT @ {K_CUT}")
plt.ylabel("Cumulative Percent Error [%]")
plt.xlabel("Time step k")
plt.title("Overall cumulative error (includes TRAIN mistakes)")
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

# 2) DEPLOY cumulative error
plt.figure(figsize=(11, 4))
plt.plot(np.arange(K_CUT+1, Np), cum_err_deploy, label=f"DEPLOY cumulative (final {cum_err_deploy[-1]:.2f}%)")
plt.axvline(K_CUT, linestyle="--", linewidth=2, label=f"CUT @ {K_CUT}")
plt.ylabel("Cumulative Percent Error [%]")
plt.xlabel("Time step k")
plt.title("DEPLOY cumulative error (denominator reset at CUT)")
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

# 3) Bar chart
fig, ax = plt.subplots(figsize=(7,4))
vals = [err_before, err_after]
labels = ["Before CUT (TRAIN)", "After CUT (DEPLOY 2 inexperts)"]
plt.bar(labels, vals, width=0.5)
for i, v in enumerate(vals):
    plt.text(i, v + 0.01, f"{v:.2f}%", ha="center")
plt.ylabel("Percent Error [%]")
plt.title("Cutoff Effect: Error Before vs After Cut")
plt.ylim(0, 2)
plt.tight_layout()
plt.show()

/tmp/ipykernel_927/2296940470.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/tmp/ipykernel_927/2296940470.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/tmp/ipykernel_927/2296940470.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# 4) Convergence c_hat_0
plt.figure(figsize=(11, 3.5))
plt.plot(c_hat_hist_0, label=r"$\hat{c}_0(k)$")
plt.axvline(K_CUT, linestyle="--", linewidth=2, label=f"CUT @ {K_CUT}")
plt.ylabel(r"$\hat{c}_0(k)$")
plt.xlabel("Time step k")
plt.title("Convergence of inexpert sensor 1 gain")
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

# 5) Convergence sigma_0^2
plt.figure(figsize=(11, 3.5))
plt.plot(sig2_hat_hist_0, label=r"$\hat{\sigma}_0^2(k)$")
plt.axvline(K_CUT, linestyle="--", linewidth=2, label=f"CUT @ {K_CUT}")
plt.ylabel(r"$\hat{\sigma}_0^2(k)$")
plt.xlabel("Time step k")
plt.title("Convergence of inexpert sensor 1 variance")
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

# 6) Convergence c_hat_2
plt.figure(figsize=(11, 3.5))
plt.plot(c_hat_hist_2, label=r"$\hat{c}_2(k)$")
plt.axvline(K_CUT, linestyle="--", linewidth=2, label=f"CUT @ {K_CUT}")
plt.ylabel(r"$\hat{c}_2(k)$")
plt.xlabel("Time step k")
plt.title("Convergence of inexpert sensor 2 gain")
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

# 7) Convergence sigma_2^2
plt.figure(figsize=(11, 3.5))
plt.plot(sig2_hat_hist_2, label=r"$\hat{\sigma}_2^2(k)$")
plt.axvline(K_CUT, linestyle="--", linewidth=2, label=f"CUT @ {K_CUT}")
plt.ylabel(r"$\hat{\sigma}_2^2(k)$")
plt.xlabel("Time step k")
plt.title("Convergence of inexpert sensor 2 variance")
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

/tmp/ipykernel_927/2642114353.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/tmp/ipykernel_927/2642114353.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/tmp/ipykernel_927/2642114353.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/tmp/ipykernel_927/2642114353.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Part B — Reporting for manuscript Section 4.4

Everything below reads `Xki_all`, `gt`, `y1` and the frozen parameters. The
filter is not re-run and nothing above is modified.

In [7]:
# ---------- segment definitions ----------
TRAIN = slice(0, K_CUT + 1)
DEPLOY = slice(K_CUT + 1, Np)

cam = np.where(y1 > 0, 1, -1)      # camera decision, read straight from y1

def seg_err(pred, seg):
    return 100.0 * np.mean(pred[seg] != gt[seg])

print("Percent error by segment")
print("=" * 60)
print(f"{'Condition':<30}{'Train':>10}{'Deploy':>10}{'Full':>10}")
print("-" * 60)
print(f"{'Expert alone (camera)':<30}"
      f"{seg_err(cam, TRAIN):>9.2f}%{seg_err(cam, DEPLOY):>9.2f}%"
      f"{100 * np.mean(cam != gt):>9.2f}%")
print(f"{'Network':<30}"
      f"{seg_err(Xki_all, TRAIN):>9.2f}%{seg_err(Xki_all, DEPLOY):>9.2f}%"
      f"{100 * np.mean(Xki_all != gt):>9.2f}%")
print("=" * 60)
print(f"Training samples 1-{K_CUT + 1}   |   Deployment samples {K_CUT + 2}-{Np}")
print(f"Error counts: {int(err[TRAIN].sum())} of {K_CUT + 1} in training, "
      f"{int(err[DEPLOY].sum())} of {Np - K_CUT - 1} in deployment")

Percent error by segment
Condition                          Train    Deploy      Full
------------------------------------------------------------
Expert alone (camera)             27.18%    31.17%    28.47%
Network                            1.52%     1.00%     1.35%
Training samples 1-1251   |   Deployment samples 1252-1851
Error counts: 19 of 1251 in training, 6 of 600 in deployment


### Segment comparability

The training error is computed over samples $1$–$1251$ and the deployment
error over $1252$–$1851$. The expert row above is the check the chapter asks
for: if the camera's error on the deployment segment is close to its error on
the whole recording, the two segments are of comparable difficulty.

The table below reports what makes them differ — class balance and how often
the state changes.

In [8]:
print("Class distribution and transition count by segment")
print(f"{'Segment':<10}{'N':>7}{'Walking':>17}{'Not walking':>17}"
      f"{'Transitions':>13}{'Samples/trans':>15}")
for name, seg in [("Train", TRAIN), ("Deploy", DEPLOY), ("Full", slice(0, Np))]:
    g = gt[seg]
    n = len(g)
    n_t = int(np.sum(np.abs(np.diff(g)) > 0))
    print(f"{name:<10}{n:>7}"
          f"{np.sum(g == 1):>10} ({100 * np.mean(g == 1):>4.1f}%)"
          f"{np.sum(g == -1):>10} ({100 * np.mean(g == -1):>4.1f}%)"
          f"{n_t:>13}{n / max(n_t, 1):>15.1f}")

Class distribution and transition count by segment
Segment         N          Walking      Not walking  Transitions  Samples/trans


Train        1251       443 (35.4%)       808 (64.6%)           26           48.1
Deploy        600       240 (40.0%)       360 (60.0%)            8           75.0
Full         1851       683 (36.9%)      1168 (63.1%)           34           54.4


In [9]:
# ---------- per-class recall ----------
def recall(pred, seg, cls):
    mask = (gt[seg] == cls)
    return np.nan if mask.sum() == 0 else 100.0 * np.mean(pred[seg][mask] == cls)

print("Per-class recall [%]")
print(f"{'Condition':<28}{'Walking':>12}{'Not walking':>14}")
for name, pred, seg in [("Camera alone, train",  cam,     TRAIN),
                        ("Camera alone, deploy", cam,     DEPLOY),
                        ("Network, train",       Xki_all, TRAIN),
                        ("Network, deploy",      Xki_all, DEPLOY)]:
    print(f"{name:<28}{recall(pred, seg, 1):>12.2f}{recall(pred, seg, -1):>14.2f}")

# The camera's segment error follows from its recall and the class mix alone:
#   error = p(walking) * missrate + p(not walking) * falsealarmrate
print("\nExpert error decomposition by segment")
print(f"{'Segment':<9}{'p(walk)':>9}{'miss':>8}{'miss term':>11}"
      f"{'FA':>8}{'FA term':>10}{'sum':>9}{'measured':>10}")
for name, seg in [("Train", TRAIN), ("Deploy", DEPLOY), ("Full", slice(0, Np))]:
    p_walk = float(np.mean(gt[seg] == 1))
    miss = 1.0 - recall(cam, seg, 1) / 100.0
    fa   = 1.0 - recall(cam, seg, -1) / 100.0
    t_m, t_f = 100 * p_walk * miss, 100 * (1 - p_walk) * fa
    print(f"{name:<9}{p_walk:>9.4f}{miss:>8.4f}{t_m:>10.2f}%"
          f"{fa:>8.4f}{t_f:>9.2f}%{t_m + t_f:>8.2f}%{seg_err(cam, seg):>9.2f}%")

Per-class recall [%]
Condition                        Walking   Not walking
Camera alone, train                23.25        100.00
Camera alone, deploy               22.08        100.00
Network, train                     96.61         99.50
Network, deploy                    97.92         99.72

Expert error decomposition by segment
Segment    p(walk)    miss  miss term      FA   FA term      sum  measured
Train       0.3541  0.7675     27.18%  0.0000     0.00%   27.18%    27.18%
Deploy      0.4000  0.7792     31.17%  0.0000     0.00%   31.17%    31.17%
Full        0.3690  0.7716     28.47%  0.0000     0.00%   28.47%    28.47%


In [10]:
# ---------- where the errors sit relative to state transitions ----------
W = 3   # half-width of the transition window, in samples

print(f"Error position relative to transitions (window +/- {W} samples)\n")
print(f"{'Segment':<10}{'N':>7}{'Trans':>8}{'Errors':>9}{'near':>8}"
      f"{'steady':>9}{'steady err':>14}{'err/trans':>12}")
for name, seg in [("Train", TRAIN), ("Deploy", DEPLOY)]:
    g, p = gt[seg], Xki_all[seg]
    tidx = np.where(np.abs(np.diff(g)) > 0)[0] + 1
    near = np.zeros(len(g), dtype=bool)
    for t in tidx:
        near[max(0, t - W):min(len(g), t + W + 1)] = True
    e = (p != g)
    print(f"{name:<10}{len(g):>7}{len(tidx):>8}{e.sum():>9}{e[near].sum():>8}"
          f"{e[~near].sum():>9}{100 * e[~near].mean():>13.2f}%"
          f"{e.sum() / max(len(tidx), 1):>12.2f}")

Error position relative to transitions (window +/- 3 samples)

Segment         N   Trans   Errors    near   steady    steady err   err/trans
Train        1251      26       19      11        8         0.75%        0.73
Deploy        600       8        6       4        2         0.37%        0.75


In [11]:
for cut in [50, 100, 200]:
    print(f"errors before k={cut}: {int(err[:cut].sum())} of {int(err[TRAIN].sum())}")

errors before k=50: 1 of 19
errors before k=100: 4 of 19
errors before k=200: 6 of 19


In [12]:
# ---------- Figure: cumulative error with the expert baseline ----------
cum_cam = 100.0 * np.cumsum((cam != gt).astype(int)) / (np.arange(Np) + 1)

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(cum_cam, linestyle="--", color="gray", linewidth=1.6,
        label=f"Expert alone ({cum_cam[-1]:.2f}%)")
ax.plot(cum_err_overall, color="steelblue", linewidth=2,
        label=f"Network ({cum_err_overall[-1]:.2f}%)")
ax.axvline(K_CUT, linestyle="--", color="firebrick", linewidth=1.6,
           label=f"Expert withdrawn, $k_c$ = {K_CUT}")
ax.set_xlabel("Time step $k$")
ax.set_ylabel("Cumulative error [%]")
ax.set_yscale("log")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.savefig("ch5_binary_cumulative.pdf", bbox_inches="tight")
plt.show()

/tmp/ipykernel_927/2433438375.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
# ---------- Figure: per-class recall, expert against network ----------
fig, ax = plt.subplots(figsize=(7.2, 3.8))
idx = np.arange(2)
w = 0.2
series = [("Expert alone, training",   cam,     TRAIN,  "#9e9e9e"),
          ("Expert alone, deployment", cam,     DEPLOY, "#616161"),
          ("Network, training",        Xki_all, TRAIN,  "steelblue"),
          ("Network, deployment",      Xki_all, DEPLOY, "darkorange")]

for j, (label, pred, seg, colour) in enumerate(series):
    vals = [recall(pred, seg, 1), recall(pred, seg, -1)]
    off = (j - 1.5) * w
    ax.bar(idx + off, vals, w, label=label, color=colour)
    for i, v in enumerate(vals):
        ax.text(i + off, v + 1.2, f"{v:.1f}", ha="center", fontsize=7)

ax.set_xticks(idx)
ax.set_xticklabels(["Walking", "Not walking"])
ax.set_ylabel("Recall [%]")
ax.set_ylim(0, 118)
ax.legend(fontsize=7.5, loc="lower center", ncol=2, framealpha=0.9)
plt.tight_layout()
plt.savefig("ch5_binary_recall.pdf", bbox_inches="tight")
plt.show()

/tmp/ipykernel_927/767409076.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
# ---------- Figure: parameter convergence, both microphones together ----------
fig, ax = plt.subplots(2, 1, sharex=True, figsize=(9, 5.5))

ax[0].plot(c_hat_hist_0, color="steelblue",  label=r"$\hat{c}_1(k)$  mic 1")
ax[0].plot(c_hat_hist_2, color="darkorange", label=r"$\hat{c}_2(k)$  mic 2")
ax[0].axvline(K_CUT, linestyle="--", color="firebrick", linewidth=1.6)
ax[0].set_ylabel("Coupling")
ax[0].legend(loc="lower right", fontsize=9)

ax[1].plot(sig2_hat_hist_0, color="steelblue",  label=r"$\hat{\sigma}_1^2(k)$  mic 1")
ax[1].plot(sig2_hat_hist_2, color="darkorange", label=r"$\hat{\sigma}_2^2(k)$  mic 2")
ax[1].axvline(K_CUT, linestyle="--", color="firebrick", linewidth=1.6,
              label=f"$k_c$ = {K_CUT}")
ax[1].set_ylabel("Variance")
ax[1].set_xlabel("Time step $k$")
ax[1].legend(loc="lower right", fontsize=9)

plt.tight_layout()
plt.savefig("ch5_binary_params.pdf", bbox_inches="tight")
plt.show()

/tmp/ipykernel_927/2440635817.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
# ---------- paste-ready LaTeX ----------
print("% ---- Table 6 row ----")
print(f"    Binary walking detection & ${Np}$ & ${K_CUT}$ & Fused \\\\")

print("\n% ---- Table 7: binary walking detection across the cutoff ----")
print(r"  \begin{tabular}{@{}lccc@{}}")
print(r"    \toprule")
print(r"    Condition & Sensors in the filter & Samples & Error \\")
print(r"    \midrule")
print(f"    Expert alone, full recording & Expert only & $1$--${Np}$ & "
      f"${100 * np.mean(cam != gt):.2f}\\%$ \\\\")
print(f"    Expert alone, deployment segment & Expert only & "
      f"${K_CUT + 2}$--${Np}$ & ${seg_err(cam, DEPLOY):.2f}\\%$ \\\\")
print(f"    Training, $k \\le k_c$ & Expert $+$ two inexpert & "
      f"$1$--${K_CUT + 1}$ & ${seg_err(Xki_all, TRAIN):.2f}\\%$ \\\\")
print(f"    Deployment, $k > k_c$ & Two inexpert only & "
      f"${K_CUT + 2}$--${Np}$ & ${seg_err(Xki_all, DEPLOY):.2f}\\%$ \\\\")
print(r"    \bottomrule")
print(r"  \end{tabular}")

print("\n% ---- Table: per-class recall across the cutoff ----")
print(r"  \begin{tabular}{@{}lcc@{}}")
print(r"    \toprule")
print(r"    Condition & Walking & Not walking \\")
print(r"    \midrule")
for name, pred, seg in [("Expert alone, deployment segment", cam,     DEPLOY),
                        ("Training, $k \\le k_c$",           Xki_all, TRAIN),
                        ("Deployment, $k > k_c$",            Xki_all, DEPLOY)]:
    print(f"    {name} & ${recall(pred, seg, 1):.2f}\\%$ & "
          f"${recall(pred, seg, -1):.2f}\\%$ \\\\")
print(r"    \bottomrule")
print(r"  \end{tabular}")

print("\n% ---- frozen parameters at k_c ----")
print(f"$\\hat{{c}}_1(k_c) = {c_frozen_0:.4f}$, "
      f"$\\hat{{\\sigma}}_1^2(k_c) = {sig2_frozen_0:.4f}$, "
      f"$\\hat{{c}}_2(k_c) = {c_frozen_2:.4f}$, "
      f"$\\hat{{\\sigma}}_2^2(k_c) = {sig2_frozen_2:.4f}$")

% ---- Table 6 row ----
    Binary walking detection & $1851$ & $1250$ & Fused \\

% ---- Table 7: binary walking detection across the cutoff ----
  \begin{tabular}{@{}lccc@{}}
    \toprule
    Condition & Sensors in the filter & Samples & Error \\
    \midrule
    Expert alone, full recording & Expert only & $1$--$1851$ & $28.47\%$ \\
    Expert alone, deployment segment & Expert only & $1252$--$1851$ & $31.17\%$ \\
    Training, $k \le k_c$ & Expert $+$ two inexpert & $1$--$1251$ & $1.52\%$ \\
    Deployment, $k > k_c$ & Two inexpert only & $1252$--$1851$ & $1.00\%$ \\
    \bottomrule
  \end{tabular}

% ---- Table: per-class recall across the cutoff ----
  \begin{tabular}{@{}lcc@{}}
    \toprule
    Condition & Walking & Not walking \\
    \midrule
    Expert alone, deployment segment & $22.08\%$ & $100.00\%$ \\
    Training, $k \le k_c$ & $96.61\%$ & $99.50\%$ \\
    Deployment, $k > k_c$ & $97.92\%$ & $99.72\%$ \\
    \bottomrule
  \end{tabular}

% ---- frozen parameters at k_c ---